### This notebook compute "P5. Glacial retreat rate or index" indicator for the 27 basins of IKI Project

Spanish: Indice o tasa de retroceso glaciar

**Created:** 10/2025 by Serena Gilson. Current contact: Sophia Bakar (sbakar@rti.org)

**Assumptions:** Replaced any negative values (indicating glaciar growth, assuming that the technology just didn't pick up growth). 

**N/A v 0 Handling:** 
COMIDs with no glacier data (not in either inventory) get NaN  
COMIDs with glaciers but no loss get 0  
COMIDs with glacier loss get the calculated rate  
 
**Future work:** 
Possibly replace the glacial area with the projected shapefiles we are recieving from INAIGEM - for now, use the glacial rate as detailed in this file.

**Notes:** Could update the years with newer inventories as needed. 

Donde 1962,1955 y 2016 son los valores del contenido de masa glaciar de acuerdo al 
inventario de INAIGEM en cada año respectivamente. Cabe destacar que, en la ecuación, 
se debe considerar el año correspondiente a la imagen satelital disponible para la región
en análisis dado que, de acuerdo con el inventario de Hidrandina de 1989, las imágenes 
pueden haber sido tomadas en el año 1962 o en el año 1955. En esta ecuación, se utiliza 
el último inventario realizado en el 2018, el cual emplea imágenes satelitales del año 
2016. En caso de contar con información más actual, se recomienda utilizar ese último 
dato  

General Methodology:  
1. Calculates present and historical glacier areas by COMID.  
2. Calculates the glacier area loss rate.  
3. Uses the glacier loss rate to determine the remaining glacier area (RGl_Km2).   


In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import geopandas as gpd
from rasterstats import zonal_stats
import rasterio
import yaml
from pathlib import Path

In [ ]:
# set master path for input data from config file

config_path = Path("../../config.yaml")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

master_path = Path(config["master_path"])

In [ ]:
# set paths for input data and indicators database 
subbasins_shapefile = master_path / "Modelacion" / "Grupos_Modelacion" / "GIS_WaterALLOC_General" / "Peru_AHD_with_districts.shp"
db_path = master_path / "Indicadores" / "BD_RiesgoClimatico_IKI.db"
glaciares_actual = master_path / "Indicadores" / "Datos_INAIGEM" / "Glaciares_2020_COMID_v2.shp"
glaciares_historico = master_path / "Indicadores" / "Datos_INAIGEM" / "Glaciares_1989_COMID_v2.shp"

In [ ]:
IndID= 105 #Indicator ID (Exposure = 2 + 0X where X is the Exposure Indicator number, Peligro= 1 +0x, VSB= 3 +0x, VSS= 4 +0x, VCA= 5 +0x)
# get scenarios from database 
conn = sqlite3.connect(db_path)

scenarios_df = pd.read_sql_query(
    """
    SELECT ScnID, ScnName
    FROM ScnMod
    """,
    conn
)

conn.close()

# For now, only populate baseline and first future, in future use all scenarios listed in the database and comment out the lines below
scenario_ids = scenarios_df.loc[
    scenarios_df['ScnID'].isin([1, 2]), 'ScnID'
].tolist()

# for all scenarios:
# scenario_ids = scenarios_df['ScnID'].tolist()

In [5]:
# update for additional future scenarios as needed 
scenario_glacier_config = {
    1: {
        "simulated_year": 2020,
        "use_future_glaciers": False
    },
    2: {
        "simulated_year": 2050,
        "use_future_glaciers": False  # change to True later when we have a future glacier dataset
    }
}

In [ ]:
# define years 
actual_year= 2020
historico_year= 1962 #the inventory was processed in 1989 but the images are from 1962

# Load the shapefiles
subbasins_gdf = gpd.read_file(subbasins_shapefile).to_crs('EPSG:32718')
glaciares_actual_gdf = gpd.read_file(glaciares_actual).to_crs('EPSG:32718')
glaciares_historico_gdf = gpd.read_file(glaciares_historico).to_crs('EPSG:32718')



In [ ]:
# Function to calculate glacier loss and remaining area 
def compute_glacier_metrics(subbasins_gdf, glaciares_actual_gdf, glaciares_historico_gdf, simulated_year,
                            actual_year=actual_year, historico_year=historico_year):
    # Sum areas by COMID
    glaciares_actual_summed = (
        glaciares_actual_gdf.groupby('COMID')['AreaKm2'].sum().reset_index().rename(columns={'AreaKm2': 'AreaKm2_2020'})
    )
    glaciares_historico_summed = (
        glaciares_historico_gdf.groupby('COMID')['AreaKm2'].sum().reset_index().rename(columns={'AreaKm2': 'AreaKm2_1989'})
    )

    # Merge
    glaciares_merged = glaciares_actual_summed.merge(glaciares_historico_summed, on='COMID', how='outer').fillna(0)

    # Glacier loss rate
    glaciares_merged['Glacier_Loss_Rate_Km2_per_year'] = (
        (glaciares_merged['AreaKm2_1989'] - glaciares_merged['AreaKm2_2020']) /
        (actual_year - historico_year)
    ).clip(lower=0)

    # Remaining glacier area
    glaciares_merged['RGl_Km2'] = (
        glaciares_merged['AreaKm2_2020'] -
        glaciares_merged['Glacier_Loss_Rate_Km2_per_year'] * (simulated_year - actual_year)
    ).clip(lower=0).round(2)

    # Merge with subbasins
    out = subbasins_gdf.merge(
        glaciares_merged[['COMID', 'Glacier_Loss_Rate_Km2_per_year', 'RGl_Km2']],
        on='COMID',
        how='left'
    )

    return out

In [19]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

insert_query = """
INSERT OR REPLACE INTO IndValues_Dyn (ScnID, IndID, COMID, Value)
VALUES (?, ?, ?, ?);
"""

rows_to_insert = []

for scn_id in scenario_ids:
    cfg = scenario_glacier_config[scn_id]

    # Compute glacier metrics (currently same for baseline and first future)
    subbasins_with_loss = compute_glacier_metrics(
        subbasins_gdf=subbasins_gdf,
        glaciares_actual_gdf=glaciares_actual_gdf,
        glaciares_historico_gdf=glaciares_historico_gdf,
        simulated_year=cfg["simulated_year"],
        actual_year=actual_year,
        historico_year=historico_year
    )

    for _, row in subbasins_with_loss.iterrows():
        rows_to_insert.append((
            scn_id,
            IndID,
            int(row['COMID']),
            row['RGl_Km2']
        ))


In [20]:
# check that min and max values match the expected range based on the Indicators Table 
indicator_limits = pd.read_sql_query(
    """
    SELECT IndID, Min, Max
    FROM Indicators
    WHERE IndID = ?
    """,
    conn,
    params=(IndID,)
)

if indicator_limits.empty:
    raise ValueError(f"No entry found in Indicators table for IndID = {IndID}")

ind_min = indicator_limits.loc[0, 'Min']
ind_max = indicator_limits.loc[0, 'Max']

print(f"Indicator {IndID}  Min: {ind_min}, Max: {ind_max}")

value_stats = (
    subbasins_with_loss['RGl_Km2']
    .agg(['min', 'max', 'count'])
    .reset_index()
)

print("\n=== Values to be inserted (by scenario) ===")
print(value_stats)

# check for duplicates
df_check = pd.DataFrame(rows_to_insert, columns=['ScnID', 'IndID', 'COMID', 'Value'])
duplicates = df_check.duplicated(subset=['ScnID', 'IndID', 'COMID'])
print("Duplicates in rows_to_insert:", df_check[duplicates])

Indicator 105  Min: 0, Max: 400

=== Values to be inserted (by scenario) ===
   index  RGl_Km2
0    min     0.00
1    max    35.57
2  count   330.00
Duplicates in rows_to_insert: Empty DataFrame
Columns: [ScnID, IndID, COMID, Value]
Index: []


In [21]:
#Execute insert to SQLite Database
cursor.executemany(insert_query, rows_to_insert)
conn.commit()
conn.close()